# 21. RL for diffusion generation — DDPO

Stable-Diffusion-v1-style conditional U-Net, 1000-step scaled-linear base schedule, 50-step stochastic DDIM rollout, classifier-free guidance, transition log-probabilities, terminal reward/advantage, PPO-clipped DDPO update를 작은 tensor로 실행한다.


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


torch.manual_seed(4)
torch.set_num_threads(min(2, torch.get_num_threads()))
device = torch.device("cpu")
print("device:", device)


## 1. Stable-Diffusion-v1-style conditional U-Net

Down path는 세 cross-attention stage와 한 plain stage를 사용하고, mid cross-attention 뒤 decoder는 한 plain up stage와 세 cross-attention up stage를 사용한다. Down stage마다 residual block 2개, up stage마다 3개를 사용한다.


In [ ]:
def timestep_embedding(timestep, dim, max_period=10000):
    half = dim // 2
    frequencies = torch.exp(
        -math.log(max_period)
        * torch.arange(half, device=timestep.device, dtype=torch.float32)
        / half
    )
    arguments = timestep.float()[:, None] * frequencies[None]
    embedding = torch.cat([arguments.cos(), arguments.sin()], dim=-1)
    if dim % 2:
        embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)
    return embedding

def group_norm(channels):
    groups = min(32, channels)
    while channels % groups != 0:
        groups -= 1
    return nn.GroupNorm(groups, channels, eps=1e-5)

class ResBlock2D(nn.Module):
    def __init__(self, input_channels, output_channels, time_dim):
        super().__init__()
        self.norm1 = group_norm(input_channels)
        self.conv1 = nn.Conv2d(input_channels, output_channels, 3, padding=1)
        self.time_projection = nn.Linear(time_dim, output_channels)
        self.norm2 = group_norm(output_channels)
        self.conv2 = nn.Conv2d(output_channels, output_channels, 3, padding=1)
        self.skip = nn.Identity() if input_channels == output_channels else nn.Conv2d(input_channels, output_channels, 1)
    def forward(self, x, time_embedding_value):
        hidden = self.conv1(F.silu(self.norm1(x)))
        hidden = hidden + self.time_projection(F.silu(time_embedding_value))[:, :, None, None]
        hidden = self.conv2(F.silu(self.norm2(hidden)))
        return hidden + self.skip(x)

class GEGLU(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.projection = nn.Linear(dim, 2 * hidden_dim)
        self.output = nn.Linear(hidden_dim, dim)
    def forward(self, x):
        value, gate = self.projection(x).chunk(2, dim=-1)
        return self.output(value * F.gelu(gate))

class SpatialTransformer(nn.Module):
    def __init__(self, channels, context_dim=32, heads=8):
        super().__init__()
        assert channels % heads == 0
        self.channels = channels
        self.norm = group_norm(channels)
        self.input_projection = nn.Conv2d(channels, channels, 1)
        self.norm_self = nn.LayerNorm(channels)
        self.self_attention = nn.MultiheadAttention(channels, heads, batch_first=True)
        self.norm_cross = nn.LayerNorm(channels)
        self.cross_attention = nn.MultiheadAttention(channels, heads, kdim=context_dim, vdim=context_dim, batch_first=True)
        self.norm_ffn = nn.LayerNorm(channels)
        self.ffn = GEGLU(channels, 4 * channels)
        self.output_projection = nn.Conv2d(channels, channels, 1)
    def forward(self, x, context):
        residual = x
        hidden = self.input_projection(self.norm(x))
        batch, channels, height, width = hidden.shape
        hidden = hidden.flatten(2).transpose(1, 2)
        normalized = self.norm_self(hidden)
        attended, _ = self.self_attention(normalized, normalized, normalized, need_weights=False)
        hidden = hidden + attended
        normalized = self.norm_cross(hidden)
        attended, _ = self.cross_attention(normalized, context, context, need_weights=False)
        hidden = hidden + attended
        hidden = hidden + self.ffn(self.norm_ffn(hidden))
        hidden = hidden.transpose(1, 2).reshape(batch, channels, height, width)
        return residual + self.output_projection(hidden)

class DownStage(nn.Module):
    def __init__(self, input_channels, output_channels, time_dim, context_dim, cross_attention, add_downsample):
        super().__init__()
        self.cross_attention = cross_attention
        self.resnets = nn.ModuleList([ResBlock2D(input_channels, output_channels, time_dim), ResBlock2D(output_channels, output_channels, time_dim)])
        self.attentions = nn.ModuleList([SpatialTransformer(output_channels, context_dim, heads=8), SpatialTransformer(output_channels, context_dim, heads=8)]) if cross_attention else None
        self.downsample = nn.Conv2d(output_channels, output_channels, 3, stride=2, padding=1) if add_downsample else None
    def forward(self, hidden, time_embedding_value, context, skips):
        for block_index, resnet in enumerate(self.resnets):
            hidden = resnet(hidden, time_embedding_value)
            if self.attentions is not None:
                hidden = self.attentions[block_index](hidden, context)
            skips.append(hidden)
        if self.downsample is not None:
            hidden = self.downsample(hidden)
            skips.append(hidden)
        return hidden

class UpStage(nn.Module):
    def __init__(self, hidden_channels, skip_channels, output_channels, time_dim, context_dim, cross_attention, add_upsample):
        super().__init__()
        self.cross_attention = cross_attention
        self.resnets = nn.ModuleList()
        self.attentions = nn.ModuleList() if cross_attention else None
        current_channels = hidden_channels
        for skip_channel in skip_channels:
            self.resnets.append(ResBlock2D(current_channels + skip_channel, output_channels, time_dim))
            current_channels = output_channels
            if self.attentions is not None:
                self.attentions.append(SpatialTransformer(output_channels, context_dim, heads=8))
        self.upsample = nn.Sequential(nn.Upsample(scale_factor=2.0, mode="nearest"), nn.Conv2d(output_channels, output_channels, 3, padding=1)) if add_upsample else None
    def forward(self, hidden, time_embedding_value, context, skips):
        for block_index, resnet in enumerate(self.resnets):
            skip = skips.pop()
            if hidden.shape[-2:] != skip.shape[-2:]:
                hidden = F.interpolate(hidden, size=skip.shape[-2:], mode="nearest")
            hidden = torch.cat([hidden, skip], dim=1)
            hidden = resnet(hidden, time_embedding_value)
            if self.attentions is not None:
                hidden = self.attentions[block_index](hidden, context)
        if self.upsample is not None:
            hidden = self.upsample(hidden)
        return hidden

class SmallWidthStableDiffusionUNet(nn.Module):
    def __init__(self, input_channels=4, context_dim=32, block_channels=(8, 16, 32, 32), time_dim=32):
        super().__init__()
        self.block_channels = tuple(block_channels)
        self.time_dim = time_dim
        self.input_conv = nn.Conv2d(input_channels, block_channels[0], 3, padding=1)
        self.time_mlp = nn.Sequential(nn.Linear(time_dim, 4 * time_dim), nn.SiLU(), nn.Linear(4 * time_dim, time_dim))
        down_specs = [
            (block_channels[0], block_channels[0], True, True),
            (block_channels[0], block_channels[1], True, True),
            (block_channels[1], block_channels[2], True, True),
            (block_channels[2], block_channels[3], False, False),
        ]
        self.down_stages = nn.ModuleList([DownStage(stage_input, stage_output, time_dim, context_dim, has_cross_attention, add_downsample) for stage_input, stage_output, has_cross_attention, add_downsample in down_specs])
        deepest = block_channels[-1]
        self.mid_resnet1 = ResBlock2D(deepest, deepest, time_dim)
        self.mid_attention = SpatialTransformer(deepest, context_dim, heads=8)
        self.mid_resnet2 = ResBlock2D(deepest, deepest, time_dim)
        skip_channels = [block_channels[0]]
        for stage_index, stage_output in enumerate(block_channels):
            skip_channels.extend([stage_output, stage_output])
            if stage_index < 3:
                skip_channels.append(stage_output)
        up_output_channels = [block_channels[3], block_channels[2], block_channels[1], block_channels[0]]
        up_cross_attention = [False, True, True, True]
        self.up_stages = nn.ModuleList()
        hidden_channels = block_channels[-1]
        for stage_index, output_channels in enumerate(up_output_channels):
            stage_skip_channels = [skip_channels.pop(), skip_channels.pop(), skip_channels.pop()]
            self.up_stages.append(UpStage(hidden_channels, stage_skip_channels, output_channels, time_dim, context_dim, up_cross_attention[stage_index], stage_index < 3))
            hidden_channels = output_channels
        assert not skip_channels
        self.output_norm = group_norm(block_channels[0])
        self.output_conv = nn.Conv2d(block_channels[0], input_channels, 3, padding=1)
    def forward(self, latent, timestep, context):
        time = self.time_mlp(timestep_embedding(timestep, self.time_dim))
        hidden = self.input_conv(latent)
        skips = [hidden]
        for down_stage in self.down_stages:
            hidden = down_stage(hidden, time, context, skips)
        hidden = self.mid_resnet1(hidden, time)
        hidden = self.mid_attention(hidden, context)
        hidden = self.mid_resnet2(hidden, time)
        for up_stage in self.up_stages:
            hidden = up_stage(hidden, time, context, skips)
        assert not skips
        return self.output_conv(F.silu(self.output_norm(hidden)))

unet = SmallWidthStableDiffusionUNet().to(device)
assert [stage.cross_attention for stage in unet.down_stages] == [True, True, True, False]
assert [stage.cross_attention for stage in unet.up_stages] == [False, True, True, True]
assert all(len(stage.resnets) == 2 for stage in unet.down_stages)
assert all(len(stage.resnets) == 3 for stage in unet.up_stages)
print("UNet down/up stages:", len(unet.down_stages), len(unet.up_stages))


## 2. 1000-step base schedule and 50-step stochastic DDIM policy

Base scheduler는 Stable Diffusion v1의 scaled-linear beta range를 사용하고 inference는 50 DDIM timestep을 사용한다. `eta > 0`이면 각 reverse step은 Gaussian transition distribution이 된다.


In [ ]:
class DDIMPolicyScheduler:
    def __init__(self, num_train_timesteps=1000, num_inference_steps=50, beta_start=0.00085, beta_end=0.012, steps_offset=1, set_alpha_to_one=False):
        self.num_train_timesteps = num_train_timesteps
        self.num_inference_steps = num_inference_steps
        self.steps_offset = steps_offset
        self.set_alpha_to_one = set_alpha_to_one
        self.betas = torch.linspace(math.sqrt(beta_start), math.sqrt(beta_end), num_train_timesteps, dtype=torch.float32).square()
        self.alphas = 1.0 - self.betas
        self.alpha_cumprod = torch.cumprod(self.alphas, dim=0)
        self.final_alpha_cumprod = torch.tensor(1.0) if set_alpha_to_one else self.alpha_cumprod[0]
        step_ratio = num_train_timesteps // num_inference_steps
        timesteps = torch.arange(num_inference_steps) * step_ratio
        self.timesteps = timesteps.flip(0) + steps_offset
        self.step_ratio = step_ratio
    def transition_mean_sigma(self, predicted_epsilon, timestep, sample, eta=1.0):
        alpha_cumprod = self.alpha_cumprod.to(sample.device)
        alpha_t = alpha_cumprod[timestep]
        previous_timestep = timestep - self.step_ratio
        alpha_previous = alpha_cumprod[previous_timestep] if previous_timestep >= 0 else self.final_alpha_cumprod.to(sample.device)
        beta_t = 1 - alpha_t
        predicted_x0 = (sample - beta_t.sqrt() * predicted_epsilon) / alpha_t.sqrt()
        variance = (1 - alpha_previous) / (1 - alpha_t) * (1 - alpha_t / alpha_previous)
        std_dev = eta * variance.clamp_min(1e-20).sqrt()
        direction = (1 - alpha_previous - std_dev.square()).clamp_min(0).sqrt() * predicted_epsilon
        mean = alpha_previous.sqrt() * predicted_x0 + direction
        return mean, std_dev

def gaussian_log_probability(sample, mean, sigma):
    log_probability = -((sample.detach() - mean) ** 2) / (2 * sigma.square()) - torch.log(sigma) - 0.5 * math.log(2 * math.pi)
    return log_probability.mean(dim=tuple(range(1, log_probability.ndim)))

scheduler = DDIMPolicyScheduler()
assert scheduler.num_train_timesteps == 1000
assert scheduler.num_inference_steps == 50
assert len(scheduler.timesteps) == 50
assert int(scheduler.timesteps[0]) == 981
assert int(scheduler.timesteps[-1]) == 1
print("DDIM timesteps:", int(scheduler.timesteps[0]), "...", int(scheduler.timesteps[-1]))


## 3. Old-policy rollout with classifier-free guidance and per-step log-probabilities

각 sampled reverse transition의 probability는 sampling에 사용한 guided denoiser와 같은 policy mean으로 계산한다.


In [ ]:
old_policy = SmallWidthStableDiffusionUNet().to(device)
current_policy = SmallWidthStableDiffusionUNet().to(device)
current_policy.load_state_dict(old_policy.state_dict())
batch_size = 2
conditional_context = torch.randn(batch_size, 3, 32, device=device)
unconditional_context = torch.zeros_like(conditional_context)
guidance_scale = 5.0
latent = torch.randn(batch_size, 4, 8, 8, device=device)

def classifier_free_guided_epsilon(policy, latent_state, timestep, conditional_context, unconditional_context, guidance_scale):
    unconditional_epsilon = policy(latent_state, timestep, unconditional_context)
    conditional_epsilon = policy(latent_state, timestep, conditional_context)
    return unconditional_epsilon + guidance_scale * (conditional_epsilon - unconditional_epsilon)

states = []
next_states = []
timesteps = []
old_log_probs = []
with torch.no_grad():
    for timestep_value in scheduler.timesteps.tolist():
        timestep = torch.full((batch_size,), int(timestep_value), dtype=torch.long, device=device)
        predicted_epsilon = classifier_free_guided_epsilon(old_policy, latent, timestep, conditional_context, unconditional_context, guidance_scale)
        mean, sigma = scheduler.transition_mean_sigma(predicted_epsilon, int(timestep_value), latent, eta=1.0)
        next_latent = mean + sigma * torch.randn_like(latent)
        log_prob = gaussian_log_probability(next_latent, mean, sigma)
        states.append(latent.clone())
        next_states.append(next_latent.clone())
        timesteps.append(int(timestep_value))
        old_log_probs.append(log_prob.clone())
        latent = next_latent
final_latent = latent
old_log_probs = torch.stack(old_log_probs, dim=1)
assert old_log_probs.shape == (batch_size, 50)
assert torch.isfinite(old_log_probs).all()
print("stored guided DDPO transitions:", old_log_probs.shape)


## 4. Terminal reward, normalized advantage, and DDPO importance-sampling update

Sampled transition은 detach하고 gradient는 current policy의 DDIM transition mean을 통해 흐른다. 동일한 terminal advantage를 각 denoising action에 사용하며 importance ratio는 PPO 방식으로 clipping한다.


In [ ]:
target = torch.zeros_like(final_latent)
reward = -(final_latent - target).square().mean(dim=(1, 2, 3))
advantage = (reward - reward.mean()) / (reward.std(unbiased=False) + 1e-6)
optimizer = torch.optim.AdamW(current_policy.parameters(), lr=1e-4)
optimizer.zero_grad()
clip_range = 0.2
loss_values = []
ratio_values = []
for transition_index, timestep_value in enumerate(timesteps):
    state = states[transition_index].detach()
    sampled_next = next_states[transition_index].detach()
    timestep = torch.full((batch_size,), timestep_value, dtype=torch.long, device=device)
    predicted_epsilon = classifier_free_guided_epsilon(current_policy, state, timestep, conditional_context, unconditional_context, guidance_scale)
    mean, sigma = scheduler.transition_mean_sigma(predicted_epsilon, timestep_value, state, eta=1.0)
    new_log_prob = gaussian_log_probability(sampled_next, mean, sigma)
    ratio = torch.exp(new_log_prob - old_log_probs[:, transition_index].detach())
    unclipped = ratio * advantage
    clipped = ratio.clamp(1.0 - clip_range, 1.0 + clip_range) * advantage
    transition_loss = -torch.minimum(unclipped, clipped).mean()
    (transition_loss / len(timesteps)).backward()
    loss_values.append(transition_loss.detach())
    ratio_values.append(ratio.detach())
optimizer.step()
mean_loss = torch.stack(loss_values).mean()
mean_ratio = torch.stack(ratio_values).mean()
assert current_policy.input_conv.weight.grad is not None
assert len(timesteps) == 50
assert guidance_scale == 5.0
print("reward:", reward)
print("mean DDPO loss:", mean_loss.item())
print("mean importance ratio:", mean_ratio.item())
